# 00 · Getting started — set up FaultyCat & prepare the lab

This one takes you from a freshly plugged-in board to a verified, ready-to-work
session: **detect → connect → verify communication → understand the channels →
wiring & safety**. Run it once whenever you sit down at the bench.

> This assumes the package is already installed — see the repo README if it
> isn't. Everything in these notebooks is real usage, nothing faked.

> ⚠️ **Safety.** FaultyCat injects faults for a living. EMFI fires high voltage
> through the coil; crowbar shorts the target's power line. **Keep the plastic
> shield installed** and never touch the exposed HV circuitry. This notebook
> only *reads* status — it never fires.

In [ ]:
import faultycat as fc
print('faultycat', fc.__version__)

## 1 · Detect the board

FaultyCat shows up as a USB composite device (VID `0x1209` / PID `0xFA17`)
that exposes **four** CDC serial interfaces. If nothing turns up, the usual
suspects are a charge-only USB cable or an unpowered board — check both.

In [ ]:
from serial.tools import list_ports
hits = [p for p in list_ports.comports() if p.vid == 0x1209 and p.pid == 0xFA17]
if hits:
    print(f'FaultyCat detected — {len(hits)} CDC interfaces:')
    for p in sorted(hits, key=lambda p: p.device):
        print(f'  {p.device}  {p.interface or ""}')
else:
    print('No board found. You can still explore with the simulator (next cell).')

## 2 · Connect

`fc.connect()` finds each CDC by its VID:PID and opens EMFI, crowbar, and the
scanner for you. **No board on hand?** Set `SIM = True` to drive an in-memory
FaultyCat that speaks the real protocols — every notebook here runs on it just
as happily as on hardware.

In [ ]:
SIM = False                      # True = no hardware, in-memory simulator
cat = fc.connect(simulator=SIM)
cat                              # renders which engines came up

## 3 · Verify communication

Now read each engine's status straight off the hardware. `state=IDLE, err=NONE`
on both means the board is healthy and sitting idle, ready to go. Think of a
clean read here as your smoke test that host↔firmware framing is working.

In [ ]:
print('EMFI   :', dict(cat.emfi.status.as_rows()))
print('CROWBAR:', dict(cat.crowbar.status.as_rows()))
print('SCANNER:', repr(cat.scanner))

## 4 · The four channels

Here's how the four CDC interfaces map to what you actually drive:

| Interface | Engine | What you drive it with |
| --- | --- | --- |
| CDC0 | **EMFI** | `cat.emfi` — electromagnetic pulse |
| CDC1 | **Crowbar** | `cat.crowbar` — voltage glitch |
| CDC2 | **Scanner shell** | `cat.scanner` (SWD/I2C/logic) + `cat.uart` control |
| CDC3 | **Target UART** | `cat.uart` data — read the target's response |

A campaign (a parameter sweep) rides the EMFI or crowbar channel:
`cat.campaign('emfi')` / `cat.campaign('crowbar')`.

## 5 · Wiring for the next notebooks

Once you know what you're attacking, wire the target to FaultyCat accordingly:

- **Glitching (crowbar)** — target **VCC** → crowbar output (LP or HP), with a common **GND**.
- **Fault injection (EMFI)** — hold the coil over the target die; no electrical contact needed.
- **Trigger (both)** — a target **GPIO that rises just before the code you want to attack** → FaultyCat **trigger input**. This is what buys you reproducible, aligned glitches via `ext_rising`.
- **SWD detection** — target **SWCLK / SWDIO / GND** → any GP channels on the **scanner header**.

From here: `01-glitching` (crowbar), `02-fault-injection` (EMFI),
`03-jtagulator-swd` (pinout).

In [ ]:
cat.close()
print('Session closed. Lab ready.')